In [21]:
import pandas as pd
import sklearn

In [22]:
df = pd.read_csv('~/sms-spam-classifier/SMSSpamCollection.csv', sep='\t', header=None, names=['label', 'message'])

In [23]:
def to_vector(text, vocab):
    words = text.lower().split()      #we dont want casesensitivity free=Free
    return [words.count(w) for w in vocab]

In [24]:
#Hold back 20% of the messages so the model will only work on those it didn't see   X
 
#baseline is always predicting "ham" -> 87% accuracy by default    X

#homomorphism   word ----> word count
#------------------------------------------------------#

#Train Naive Bayes
#Evaluate with precision,recall and confusion matrix
#Try my own messages
#Readme -> explanation of everything


In [25]:
sample = df.sample(4500)

In [26]:
ham_count = (sample["label"] == 'ham').sum()
total = len(sample)
baseline = round(ham_count / total, 2)

print(f'Baseline accuracy (always ham):  {baseline}')

Baseline accuracy (always ham):  0.86


In [27]:
# homomorphism word -> count
#it works but FREE!! is still different thant Free => 

In [33]:
from sklearn.model_selection import train_test_split

train_messages, test_messages, train_labels, test_labels = train_test_split(
    df['message'], df['label'], test_size=0.2, stratify=df['label']   #stratify to maintain 87/13 ratio
)
print(len(train_messages), len(test_messages))

4457 1115


In [34]:
#fit -> learn the rules of the current data
#transform -> apply the rules (changes the current matrix, CANT CALL TRANSFORM BEFORE FIT)

from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer()  #default lowercase
X_train = vectorizer.fit_transform(train_messages)    #(row: xth message,  column: word in message ) Value apps in that message of word
X_test = vectorizer.transform(test_messages) #transform cuz we use X_train vocab in test


In [36]:
#   P(label | message)  in relation with   P(label) * P(message | label)

In [42]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report #??

model = MultinomialNB() #init
model.fit(X_train, train_labels) #probability of each X_train words, separated by label spam or ham

predictions = model.predict(X_test) #final step of assembly



In [49]:
print(f"Accuracy score: {accuracy_score(test_labels, predictions)}\n") 
print(f"Confusion matrix: {confusion_matrix(test_labels, predictions)}\n")
print(f"Classification report: {classification_report(test_labels, predictions)}")

Accuracy score: 0.9811659192825112

Confusion matrix: [[962   4]
 [ 17 132]]

Classification report:               precision    recall  f1-score   support

         ham       0.98      1.00      0.99       966
        spam       0.97      0.89      0.93       149

    accuracy                           0.98      1115
   macro avg       0.98      0.94      0.96      1115
weighted avg       0.98      0.98      0.98      1115



In [ ]:
# Example messages

In [60]:
my_custom_messages = [
    "Hey, are we still meeting for lunch tomorrow at noon?",
    "CONGRATULATIONS! You have been selected to win a free iPhone 16! Click here immediately to claim your prize.",
    "Please find attached the quarterly financial report for review before our meeting.",
    "URGENT: Your bank account has been locked. Verify your password now at htaaatp:/]/fake-bank-login.com"
]

In [61]:
X_custom = vectorizer.transform(my_custom_messages)

predictions_own = model.predict(X_custom)
probabilities_own = model.predict_proba(X_custom)

for msg, pred, prob in zip(my_custom_messages, predictions_own, probabilities_own):
    print(f"Message: '{msg}'")
    print(f"Prediction: {pred.upper()} (Confidence - Ham: {prob[0]:.2f}, Spam: {prob[1]:.2f})\n")

Message: 'Hey, are we still meeting for lunch tomorrow at noon?'
Prediction: HAM (Confidence - Ham: 1.00, Spam: 0.00)

Message: 'CONGRATULATIONS! You have been selected to win a free iPhone 16! Click here immediately to claim your prize.'
Prediction: SPAM (Confidence - Ham: 0.00, Spam: 1.00)

Message: 'Please find attached the quarterly financial report for review before our meeting.'
Prediction: HAM (Confidence - Ham: 0.98, Spam: 0.02)

Message: 'URGENT: Your bank account has been locked. Verify your password now at htaaatp:/]/fake-bank-login.com'
Prediction: SPAM (Confidence - Ham: 0.00, Spam: 1.00)



In [ ]:
#Improvements:
# less weight for words like "the" "I" "is" since they're common, more weight for "lottery" "prize" etc
# Still naive but less naive way where we consider bayes for pairs as well, "free deal" etc
#* Support Vector Machines/Logistic regression, Supposedly they have higher accuracy
